# What does the player-share model predict?

Pick one Six Nations round and inspect its forecasts. The notebook fits on all
matches before that round, then compares player shares with and without
opposition against the recency-weighted mean. The squad comparison uses prices
when the prepared roster is available.

In [43]:
from math import log
from pathlib import Path

import pandas as pd
from IPython.display import display

from six_nations_fantasy.data import load_dataset
from six_nations_fantasy.data.fantasy_fixtures import fixture_path, tournament_rounds
from six_nations_fantasy.data.load import DATA_DIR
from six_nations_fantasy.evaluation.pipeline import fit_predict, recorded_lineup
from six_nations_fantasy.evaluation.selection import prediction_lineup, scored_player_fixtures
from six_nations_fantasy.evaluation.squads import select_squad, squad_solver
from six_nations_fantasy.models import PlayerShareModel, RecencyWeightedMean
from six_nations_fantasy.models.player_share import component_units
from six_nations_fantasy.score import simple_score

data_dir = DATA_DIR   # Or Path("/path/to/your/data")
year, round_number = 2026, 1
tau = 0.75
stat = "tries"        # Choose a row from the component summary below.
team = None          # Show every side, or set a name such as "France".

In [44]:
dataset = load_dataset(data_dir)
schedule = dict(tournament_rounds(dataset.matches, year))[round_number]
round_start = schedule["date"].min()
training = dataset.matches_before(round_start)
held = dataset.select_matches(schedule["match_id"])

roster_file = fixture_path(data_dir, year)
roster = pd.read_parquet(roster_file) if roster_file.exists() else None
if roster is not None and not roster["round"].eq(round_number).any():
    roster = None
scored = scored_player_fixtures(held, round_number, roster)
fixtures = prediction_lineup(scored)
print(f"Training cutoff {round_start.date()} | {len(training.matches):,} matches")
print(f"{len(fixtures)} players from {'announced rosters' if roster is not None else 'recorded team sheets'}")

scored = scored.copy()
if "name" not in scored:
    registry_file = data_dir / "players.parquet"
    if registry_file.exists():
        registry = pd.read_parquet(registry_file)
        if "player_id" in registry:
            registry = registry.set_index("player_id")
        scored["name"] = scored["player_id"].map(registry["name"])
    else:
        scored["name"] = scored["player_id"].astype(str)

Training cutoff 2026-02-05 | 1,828 matches
138 players from announced rosters


In [45]:
share = PlayerShareModel(tau=tau, half_life_days=180)
without_opposition = PlayerShareModel(tau=0, half_life_days=180)
recency = RecencyWeightedMean(scorer=simple_score, half_life_days=180)

comparison = {
    "share": fit_predict(share, training, fixtures),
    "share_no_opposition": fit_predict(without_opposition, training, fixtures),
    "recency": fit_predict(recency, training, fixtures),
}
forecasts = pd.DataFrame(comparison).assign(actual=scored["actual"])

/home/julian/code/fantasy-rugby/six_nations_fantasy/models/player_share.py:96: UserWarning: Excluded from training: 1 incomplete team sheets (expected shirts 1–23), first: {'match_id': 276171, 'team': 'Bulls'}
  rows, team_rows, teams = full_lineups(frame, training=True)
/home/julian/code/fantasy-rugby/six_nations_fantasy/models/player_share.py:96: UserWarning: Excluded from training: 1 incomplete team sheets (expected shirts 1–23), first: {'match_id': 276171, 'team': 'Bulls'}
  rows, team_rows, teams = full_lineups(frame, training=True)


## Team multipliers across the scoring stats

For each stat, the team forecast is the historical starting average times the
competition, home/away, team and opponent multipliers. The table shows how big
the fitted opponent multipliers are in this round. It also compares the two
**refitted** team forecasts. Their difference can include changes in the team
and competition coefficients when the opponent term is removed.

In [46]:
keys = ["match_id", "team"]
recorded = recorded_lineup(held)
actual_counts = component_units(held.performances)
actual_team = actual_counts.groupby([recorded[key].to_numpy() for key in keys]).sum()
actual_team.index.names = keys

team_tables = {}
for component in share.opponent_scales:
    with_opp, _ = share.explain_component(fixtures, component)
    without_opp, _ = without_opposition.explain_component(fixtures, component)
    team_tables[component] = (
        with_opp.merge(
            without_opp[keys + ["expected_units"]].rename(
                columns={"expected_units": "expected_without_opposition"}
            ),
            on=keys, validate="one_to_one",
        )
        .join(actual_team[component].rename("observed_units"), on=keys)
        .assign(change_with_opposition=lambda rows:
                rows["expected_units"] - rows["expected_without_opposition"])
    )

summary = {}
for component, sides in team_tables.items():
    with_error = sides["expected_units"] - sides["observed_units"]
    without_error = sides["expected_without_opposition"] - sides["observed_units"]
    summary[component] = {
        "smallest opponent multiplier": sides["opponent_multiplier"].min(),
        "largest opponent multiplier": sides["opponent_multiplier"].max(),
        "mean absolute change in team units": sides["change_with_opposition"].abs().mean(),
        "team MAE with opposition": with_error.abs().mean(),
        "team MAE without opposition": without_error.abs().mean(),
    }
display(pd.DataFrame.from_dict(summary, orient="index").round(3))
display(team_tables[stat].drop(columns=["date", "competition"]).round(3))

,smallest opponent multiplier,largest opponent multiplier,mean absolute change in team units,team MAE with opposition,team MAE without opposition
tries,0.604,1.626,0.737,1.271,0.840
try_assists,0.488,1.727,0.729,1.491,1.040
conversions,0.620,1.531,0.424,0.974,0.678
penalty_goals,0.542,1.517,0.367,0.982,0.727
drop_goals,0.841,1.354,0.005,0.021,0.021
defenders_beaten,0.756,1.242,2.128,6.680,7.089
carried_meters,0.863,1.159,34.827,140.188,112.120
completed_tackles,0.872,1.132,13.010,23.088,29.352
turnovers_won,0.860,1.165,0.762,2.362,1.884
offloads,0.840,1.209,0.614,3.897,3.641


,match_id,team,opponent,home,competition_multiplier,home_multiplier,team_multiplier,opponent_multiplier,baseline_units,expected_units,expected_without_opposition,observed_units,change_with_opposition
0,290465,France,Ireland,True,0.842,1.195,1.514,0.604,3.185,2.931,5.052,5,-2.122
1,290465,Ireland,France,False,0.842,0.826,1.382,0.857,3.185,2.624,3.368,2,-0.745
2,290466,Italy,Scotland,True,0.842,1.195,0.671,1.001,3.185,2.153,2.023,2,0.129
3,290466,Scotland,Italy,False,0.842,0.826,1.004,1.626,3.185,3.617,2.344,2,1.273
4,290467,England,Wales,True,0.842,1.195,0.975,1.117,3.185,3.489,3.370,6,0.119
5,290467,Wales,England,False,0.842,0.826,0.720,1.037,3.185,1.655,1.622,1,0.033


## How the team total is split

The recency mean estimates each player’s count in their announced role. Divide by the sum for the lineup to get their share. Multiply by the predicted team total to get expected units.

In [47]:
_, allocation = share.explain_component(fixtures, stat)
actual_players = recorded[["match_id", "player_id"]].assign(
    observed_units=actual_counts[stat].to_numpy()
)
allocation = (
    allocation.assign(name=scored["name"].to_numpy())
    .merge(actual_players, on=["match_id", "player_id"], how="left", validate="one_to_one")
    .join(actual_team[stat].rename("observed_team_units"), on=keys)
)
allocation["observed_share"] = (
    allocation["observed_units"].fillna(0)
    / allocation["observed_team_units"].where(allocation["observed_team_units"].ne(0))
)
columns = [
    "team", "name", "number", "position_role", "recency_mean",
    "share", "observed_share", "expected_units", "observed_units",
]
selected = allocation if team is None else allocation.loc[allocation["team"].eq(team)]
display(selected.sort_values(["team", "share"], ascending=[True, False])[columns].round(3))

,team,name,number,position_role,recency_mean,share,observed_share,expected_units,observed_units
104,England,Tommy Freeman,13,Centre,1.084,0.156,0.167,0.543,1.0
113,England,Ben Spencer,22,Substitute,0.878,0.126,0.000,0.440,0.0
105,England,Immanuel Feyi-Waboso,14,Back Three,0.683,0.098,0.000,0.342,NaN
102,England,Henry Arundell,11,Back Three,0.610,0.088,0.500,0.305,3.0
114,England,Marcus Smith,23,Substitute,0.541,0.078,0.000,0.271,0.0
...,...,...,...,...,...,...,...,...,...
119,Wales,Adam Beard,5,Second-row,0.000,0.000,0.000,0.000,0.0
132,Wales,Tomas Francis,18,Substitute,0.000,0.000,0.000,0.000,0.0
133,Wales,Ben Carter,19,Substitute,0.000,0.000,0.000,0.000,0.0
134,Wales,Taine Plumtree,20,Substitute,0.000,0.000,0.000,0.000,0.0


## Does the model pick different players?

The first table compares player errors and rank correlation on this round. The
next shows where the share model and recency mean disagree most. When prices are
available, solve the same fantasy squad problem for each forecast and compare
the players selected, captain, supersub and points earned.

In [48]:
model_names = list(comparison)
errors = forecasts[model_names].sub(forecasts["actual"], axis=0)
ranks = forecasts[model_names].rank()
display(pd.DataFrame({
    "player RMSE": errors.pow(2).mean().pow(0.5),
    "rank correlation with recency": ranks.corr()["recency"],
}).round(3))

details = scored[["match_id", "team", "player_id", "name"]].join(forecasts)
details["effect of refitting with opposition"] = details["share"] - details["share_no_opposition"]
details["share minus recency"] = details["share"] - details["recency"]
order = details["share minus recency"].abs().sort_values(ascending=False).index
display(details.loc[order].head(15).round(2))

,player RMSE,rank correlation with recency
share,10.468,0.954
share_no_opposition,10.137,0.963
recency,10.842,1.000


,match_id,team,player_id,name,share,share_no_opposition,recency,actual,effect of refitting with opposition,share minus recency
104,290467,England,156918,Tommy Freeman,26.36,26.03,42.57,55.0,0.33,-16.21
113,290467,England,111662,Ben Spencer,20.28,19.97,34.61,3.0,0.31,-14.34
9,290465,France,190344,Matthieu Jalibert,29.42,34.81,43.24,55.0,-5.39,-13.83
8,290465,France,137493,Antoine Dupont,26.60,33.06,39.58,26.0,-6.46,-12.97
105,290467,England,233237,Immanuel Feyi-Waboso,22.20,21.88,34.55,0.0,0.32,-12.35
14,290465,France,175123,Thomas Ramos,23.29,26.62,34.64,44.0,-3.33,-11.35
10,290465,France,237223,Louis Bielle-Biarrey,22.80,30.03,34.13,57.0,-7.23,-11.33
129,290467,Wales,120767,Louis Rees-Zammit,23.41,23.11,34.68,22.0,0.30,-11.27
102,290467,England,161516,Henry Arundell,16.37,16.23,26.60,59.0,0.14,-10.23
106,290467,England,150081,Freddie Steward,15.10,15.08,24.81,21.0,0.02,-9.71


In [49]:
if roster is not None:
    solver = squad_solver()
    candidates = scored.join(forecasts[model_names]).dropna(subset=["cost"])
    squads = {name: select_squad(candidates, name, solver) for name in [*model_names, "actual"]}
    recency_ids = set(squads["recency"]["player_id"])
    optimum = squads["actual"]["actual_total"].sum()
    rows = []
    for name, squad in squads.items():
        actual = squad["actual_total"].sum()
        rows.append({
            "selection": "hindsight optimum" if name == "actual" else name,
            "players also picked by recency": len(recency_ids & set(squad["player_id"])),
            "captain": squad.loc[squad["captain"], "name"].iloc[0],
            "supersub": squad.loc[squad["supersub"], "name"].iloc[0],
            "actual points": actual,
            "% of optimum": 100 * actual / optimum if optimum else float("nan"),
        })
    display(pd.DataFrame(rows).round(2))
else:
    print("Prepare this round's fantasy roster to compare squad selections.")

,selection,players also picked by recency,captain,supersub,actual points,% of optimum
0,share,10,Matthieu Jalibert,Ben Spencer,489.0,66.99
1,share_no_opposition,11,Matthieu Jalibert,Ben Spencer,511.0,70.00
2,recency,16,Matthieu Jalibert,Ben Spencer,456.0,62.47
3,hindsight optimum,6,Henry Arundell,Nick Timoney,730.0,100.00
